# Determinism, seeds & replay

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 09 §9.4 (with §9.3) · type: concept-lab*

**The promise:** you will stop treating determinism as a parameter you *set* and start treating it as a property you *build* — by pinning, designing for nondeterminism, logging to replay, and testing statistically.

## 🧠 Why this matters

Set temperature to 0 and you get the same output every time, right? The honest answer is *usually, but not guaranteed* — and that gap is where production incidents live.

Even at temperature 0, identical requests can diverge. Floating-point arithmetic on GPUs is **non-associative**: `(a + b) + c` need not equal `a + (b + c)` at the last bits, and the order of parallel reductions depends on how your request is batched alongside others on the provider's servers. That can flip a *borderline* token — and **one flipped token changes everything after it**. A `seed` parameter helps but is documented as best-effort for exactly this reason. Above the hardware sits a bigger drift: providers update models, and an alias like `latest` can point at different weights next month. So you *engineer* reproducibility instead of assuming it.

## Objectives & prereqs

**By the end you can:**
- Demonstrate the *shape* of float non-associativity that makes temp-0 non-deterministic.
- Apply the four disciplines — **pin**, **design for**, **log to replay**, **test statistically** — concretely.
- Write a property-style test that asserts behavior across N runs instead of asserting an exact transcript.

**Prereqs:** `09-01-decoding-knobs-live.ipynb` (you know what temperature 0 *means*). Read book §9.4 (determinism) and §9.3 (log-probs & confidence) alongside this.

**Runs free & offline.** The float demo and the multi-run check are pure local Python / canned mock responses. The optional live cell needs `ANTHROPIC_API_KEY` and `COMPANION_MOCK=0` (≈ 2 short calls).

In [ ]:
# Setup — imports, env, and the MOCK switch.
import json
import os
import random
from datetime import datetime, timezone

from dotenv import load_dotenv

load_dotenv()

# MOCK=1 (default) keeps this notebook free, offline, and deterministic.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# Seed our own RNG so the *demonstration* of nondeterminism is itself reproducible.
random.seed(11)

print("MOCK mode:", MOCK, "— offline, no API key needed" if MOCK else "— LIVE, will call the API")

## The shape of the problem: summing the same floats two ways

You don't need a GPU to *see* why temp-0 isn't guaranteed deterministic. The root cause is that floating-point addition is non-associative — the result depends on the order of operations. Below we add the **exact same numbers** in two different orders and look at the last bits.

## 🔮 Predict

We will sum a list of floats left-to-right, then sum the *same* list in a different order. With real numbers the two totals are identical.

**Predict:** with IEEE-754 floats, will the two sums be *exactly* equal (`==`)? If not, where does the difference show up — the leading digits or the last bits?

In [ ]:
# A mix of large and tiny magnitudes — the classic trigger for ordering effects.
vals = [1e16, 1.0, -1e16, 1.0, 0.5, -0.5, 3.14159, 2.71828, 1e-8, -1e-8]

forward = 0.0
for v in vals:
    forward += v

shuffled = vals[:]
random.shuffle(shuffled)
reordered = 0.0
for v in shuffled:
    reordered += v

print("forward order   :", repr(forward))
print("reordered sum   :", repr(reordered))
print("exactly equal?  :", forward == reordered)
print("difference      :", abs(forward - reordered))

**What you just saw.** The two sums differ — typically only in the last bits, but they differ. Now imagine those bits are a *logit*, and two candidate tokens are within that margin of each other. The reduction order (set by server-side batching you don't control) decides which token wins. That single flip then conditions every subsequent token. This is the whole mechanism behind "temperature 0 is usually, not always, deterministic."

## The four disciplines, made concrete

Because you can't *eliminate* nondeterminism from the provider's side, you build reproducibility around it. The book names four disciplines; here is what each one looks like as code or as a rule.

### 1. Pin dated snapshots — never moving aliases

In production you call a *dated/versioned* snapshot identifier, not an alias like `latest`, so a provider model update can't silently change your behavior. You upgrade deliberately, behind your evals (Ch 22).

In [ ]:
# ⚠️ Pitfall: a moving alias is a time bomb — same code, different weights next month.
RISKY = "claude-latest"        # don't ship this: points at whatever is current
PINNED = "claude-opus-4-8"     # an explicit, versioned identifier you upgrade on purpose

print("Risky (aliased) :", RISKY, "  <-- behavior can drift under you")
print("Pinned          :", PINNED, "  <-- changes only when YOU change it")

### 2. Design for nondeterminism — validate and retry

Never write code that depends on the *exact wording* of a response. Validate structured output against a schema and retry on failure (Ch 10). Here is the pattern in miniature: a validator plus a bounded retry loop around a (mock) model that occasionally returns malformed JSON.

In [ ]:
def looks_valid(text):
    """Property we require, regardless of exact wording: parses as JSON with a 'city' key."""
    try:
        obj = json.loads(text)
    except (ValueError, TypeError):
        return False
    return isinstance(obj, dict) and "city" in obj


# Mock model: sometimes prepends chatter (the classic 'Here is the JSON:' failure).
_FLAKY = iter([
    'Sure! Here is the JSON:\n{"city": "Paris"}',  # attempt 1: not pure JSON
    '{"city": "Paris"}',                              # attempt 2: clean
])


def call_model():
    return next(_FLAKY)


result, MAX_RETRIES = None, 3
for attempt in range(1, MAX_RETRIES + 1):
    text = call_model()
    if looks_valid(text):
        result = json.loads(text)
        print(f"attempt {attempt}: valid -> {result}")
        break
    print(f"attempt {attempt}: rejected ({text!r:.40}...) — retrying")

assert result is not None, "all retries exhausted"

### 3. Log everything needed to replay

When an incident is *"the model did something weird,"* the only forensic tool is a record you can replay: model version, full prompt, sampling parameters, seed (if any), and the response. Build the record as a plain dict — Ch 23 wraps real observability around exactly this.

In [ ]:
def replay_record(*, model, prompt, params, seed, response):
    """Everything you need to reconstruct (and re-run) a single call."""
    return {
        "ts": datetime.now(timezone.utc).isoformat(),
        "model": model,           # the PINNED snapshot, not an alias
        "prompt": prompt,
        "params": params,          # temperature, top_p, max_tokens, ...
        "seed": seed,              # best-effort, but log it anyway
        "response": response,
    }


record = replay_record(
    model=PINNED,
    prompt="Return the capital of France as JSON: {\"city\": ...}",
    params={"temperature": 0.0, "max_tokens": 32},
    seed=42,
    response='{"city": "Paris"}',
)
print(json.dumps(record, indent=2))

### 4. Test statistically — assert properties across N runs, not string equality

Because the exact bytes can vary, a test that asserts an *exact transcript* is a liability (see the pitfall below). Instead, assert **properties** that must hold across many runs: "valid JSON," "contains a city," "score above threshold." Here is the statistical-test habit on 20 mock runs.

In [ ]:
# 20 mock responses: most are clean JSON, a couple carry harmless wording differences.
MOCK_RUNS = [
    '{"city": "Paris"}',
    '{"city":"Paris"}',                 # different whitespace — same meaning
    '{"city": "Paris", "country": "France"}',  # extra key — still valid for our property
] * 6 + ['{"city": "Paris"}', '{"city": "Paris"}']

valid = sum(looks_valid(r) for r in MOCK_RUNS)
print(f"valid-JSON-with-city across {len(MOCK_RUNS)} runs: {valid}/{len(MOCK_RUNS)}")

# A property-style assertion: ALL runs satisfy the contract, even though the exact
# strings differ. This stays green across whitespace/extra-key drift; an exact-match
# assertion would have gone red on run #2.
assert all(looks_valid(r) for r in MOCK_RUNS), "a run failed the JSON-with-city property"
print("PASS — property held across all runs (not asserting exact bytes)")

## ⚠️ Pitfall: the exact-transcript test suite

The classic failure mode: a test suite that asserts exact model outputs. It's green for weeks — then red *everywhere* after a provider model update, or *flaky* because batching nondeterminism flips one token once a day. Teams respond by pinning tests to recorded responses and quietly **stop testing the real model at all**. Test *properties and distributions*, not transcripts; keep a small recorded-response layer only for fast unit tests of *your* code around the model.

## 🔮 Predict, then run: does a `seed` make two live calls byte-identical?

**Predict:** if a provider exposes a `seed` parameter and you send two identical requests with the same seed at temperature 0, will the two responses be *byte-for-byte* identical?

Decide, then run. In `MOCK=1` we replay a canned pair that illustrates the realistic answer; with `COMPANION_MOCK=0` it issues two seeded calls.

In [ ]:
PROMPT = "Name three primary colors, comma-separated."
MODEL = PINNED


def two_seeded_calls():
    """Return two responses to the same seeded request (mock or live)."""
    if MOCK:
        # Canned to show the realistic outcome: usually identical, not *guaranteed*.
        return "Red, yellow, blue", "Red, yellow, blue"
    import anthropic  # imported only on the live path

    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment

    def once():
        msg = client.messages.create(
            model=MODEL,
            max_tokens=24,
            messages=[{"role": "user", "content": PROMPT}],
        )
        return "".join(b.text for b in msg.content if b.type == "text").strip()

    return once(), once()


a, b = two_seeded_calls()
print("call 1:", repr(a))
print("call 2:", repr(b))
print("byte-identical?", a == b, "— usually true, but the provider documents it as best-effort")

**What you just saw.** Same seed, same prompt — and the two outputs usually match. *Usually.* Providers describe seeded sampling as best-effort precisely because of the float/batching effects from the first demo. So you log the seed (it improves your odds and your forensics) but you never *depend* on byte-identity. Note that the latest Opus-tier snapshot here is called with no `temperature`/`seed` sampling params — those are removed on the newest models — which is itself a reminder that the determinism you control is structural (pin, validate, log, test), not a single flag.

## 🎯 Senior lens

Where an endpoint exposes **log-probabilities** (§9.3) — the model's own estimate of how likely each emitted token was — you get a usable confidence signal. Two moves turn it into leverage: **gate and escalate** low-confidence answers to a stronger model or a human (Ch 20), and feed token probabilities into **calibration analysis** in your evals (Ch 22) — checking whether stated confidence actually tracks accuracy *before* you trust a threshold in production. Logprobs won't catch a *confident* hallucination, but for the large class of genuinely-uncertain cases they let you act on uncertainty instead of being blind to it. Availability varies by provider as of early 2026 — treat it as a capability to check for, not assume.

## Recap

- **Temp-0 is usually, not always, deterministic** — GPU float non-associativity plus server-side batching can flip a borderline token, and one flip cascades.
- **Determinism is built, not set:** **pin** dated snapshots, **design for** nondeterminism (validate + retry), **log to replay** (model, prompt, params, seed, response), **test statistically**.
- **`seed` is best-effort** — log it, never depend on byte-identity.
- **Assert properties across N runs**, not exact transcripts; an exact-match suite goes red or flaky and erodes trust in real-model testing.
- **Log-probs** (where available) let you gate, escalate, and calibrate — act on uncertainty instead of being blind to it.

## Exercises

Each *changes something* and asks you to predict the result first.

1. **Amplify the flip.** Construct a list of floats where forward vs. reordered summation differ by *more* than the demo. Predict which magnitude mix maximizes the gap (hint: think large + tiny + cancellation), then verify.
2. **Break the property test on purpose.** Add a malformed entry (e.g. `'city: Paris'`) to `MOCK_RUNS` and predict whether the `all(looks_valid(...))` assertion fails. Then make the test *robust* by reporting the pass-rate instead of asserting 100%.
3. **Replay round-trip.** Serialize a `replay_record` to a JSON string, read it back, and assert every field survived. Predict which field type (if any) would *not* round-trip cleanly through `json.dumps`.

In [ ]:
# Exercise 1 — your code here.


In [ ]:
# Exercise 2 — your code here.


In [ ]:
# Exercise 3 — your code here.


## Next

- **Next notebook:** [`09-03-streaming-and-latency-anatomy.ipynb`](09-03-streaming-and-latency-anatomy.ipynb) — TTFT vs. decode rate, streaming, and the token economics of a call.
- **Book:** §9.4 (determinism) and §9.3 (log-probs); calibration analysis lands in §22.
- **Blueprint this feeds:** [`blueprints/llm-gateway/`](../../../blueprints/llm-gateway/) — which owns the usage logging and replay records sketched here (built from Ch 11).
- **Capstone:** the pinned-snapshot + per-call logging defaults the capstone `llm/` client encodes.